In [1]:
# 1. 数据基础库
%pip install numpy pandas

# 2. 可视化库
%pip install matplotlib seaborn

# 3. 机器学习库
%pip install scikit-learn

# 4. 统计分析库
%pip install statsmodels scipy

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\modeling\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\modeling\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\modeling\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\modeling\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 统计分析库
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

# 机器学习库
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 设置中文显示和图形样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
print("=" * 60)
print("         时间序列预测ARIMA模型 - 股票价格预测案例")
print("=" * 60)

第一步：数据生成与加载

In [ ]:
def generate_stock_data(start_date='2025-01-01', periods=100,initial_price=100):
    """
    生成模拟股票价格数据
    包含趋势、波动率和随机游走特征
    """
    np.random.seed(42)  # 保证结果可复现

    # 创建日期索引
    dates = pd.date_range(start=start_date, periods=periods, frq='D')

    # 生成价格序列：随机游走+微弱趋势+波动率变化
    returns = np.random.normal(0.001, 0.02,periods)  #日收益率
    trend = np.linspace(0,0.5,periods)/periods  #微弱上升趋势
    volatility = 1 + 0.3 * np.sin(np.linspace(0, 4*np.pi, periods))  #波动率周期变化

    returns = returns + trend
    returns = returns * volatility

    # 计算累积价格
    prices = [initial_price]
    for r in returns[1:]:
        prices.append(prices[-1] * (1+r))

    # 创建dataframe
    df = pd.DataFrame({
        'date':dates,
        'price':prices,
        'returns':[0] + list(np.diff(np.log(prices)))  # 对数收益率
    })
    df.set_index('date',inplace=True)

    return df


# 生成数据
print("\n1.数据生成与加载")
print('-'*30)
stock_data = generate_stock_data()
print(f"数据区间：{stock_data.index[0].strftime('%Y-%m-%d')}至{stock_data.index[-1].strftime('%Y-%m-%d')}")
print(f"数据点数: {len(stock_data)}")
print(f"价格范围: {stock_data['price'].min():.2f} - {stock_data['price'].max():.2f}")

# 显示基本统计信息
print("\n数据基本统计:")
print(stock_data.describe())

第二步 数据探索与可视化

In [ ]:
print("\n2.数据探索与可视化")
print('-'*30)

# 创建综合图标
fig, axes = plt.subplot(2, 2, figsize=(15,10))

# 价格时序图
axes[0,0].plot(stock_data.index,stock_data['price'],color='steelblue',linewidth=1.5)
axes.set_title('股票价格时序图',fontsize=14,fontweight='bold')
axes.set_ylabel('价格（元）')
axes.grid(True, alpha=0.3)

# 收益率时序图
axes[0,1].plot(stock_data.index,stock_data['returns'], color='orange',linewidth=1)
axes[0,1].axhline(y=0, color='red', linestyle='--', alpha=0.7)
axes[0,1].set_title('日收益率时序图',fontsize=14,fontweight='bold')
axes[0,1].set_ylabel('收益率')
axes[0,1].grid(True,alpha=0.3)

# 价格分布直方图
axes[1,0].hist(stock_data['price'],bins=30,color='lightblue',alpha=0.7,edgecolor='black')
axes[1,0].set_title('价格分布直方图', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('价格 (元)')
axes[1,0].set_ylabel('频数')

# 收益率分布直方图
axes[1,1].hist(stock_data['returns'], bins=30, color='lightcoral', alpha=0.7, edgecolor='black')
axes[1,1].set_title('收益率分布直方图', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('收益率')
axes[1,1].set_ylabel('频数')

plt.tight_layout()
plt.show()

# 计算一些描述性统计量
price_volatility = stock_data['returns'].std() * np.sqrt(252)  # 年化波动率
max_drawdown = ((stock_data['price']/stock_data['price'].cummax - 1)).min()  # 最大回撤

print(f"年化波动率: {price_volatility:.2%}")
print(f"最大回撤: {max_drawdown:.2%}")
print(f"收益率偏度: {stock_data['returns'].skew():.3f}")
print(f"收益率峰度: {stock_data['returns'].kurtosis():.3f}")


第三步：平稳性检验(ADF)

In [ ]:
print('\n3.平稳性检验')
print('-'*30)

def detailed_adf_test(timeseries,title='时间序列'):
    """
    详细的ADF平稳性检验
    """
    result = adfuller(timeseries.dropna())  #ADF检验 = Augmented Dickey-Fuller Test（增广迪基-富勒检验）

    print(f'\n{title} ADF检测结果:')
    print(f'ADF统计量:{result[0]:.6f}')  #result[0]:ADF统计量(越负越平稳)
    print(f'p值:{result[1]:.6f}')  #result[1]  p值
    print('临界值:')
    for key,value in  result[4].items():
        print(f'   {key}:{value:.3f}')

    if result[1] <= 0.05:
        print(f'结论: {title}是平稳的 (p值 = {result[1]:.6f} < 0.05)')
        return True
    else:
        print(f'结论: {title}是非平稳的 (p值 = {result[1]:.6f} > 0.05)')
        return False

# 检验原始价格序列
price_stationary = detailed_adf_test(stock_data['price'], '原始价格序列')
# 检验收益率序列
returns_stationary = detailed_adf_test(stock_data['returns'], '收益率序列')

# 如果价格序列平稳，则进行差分
if not price_stationary:
    stock_data['price_diff'] = stock_data['price'].diff()
    diff_stationary = detailed_adf_test(stock_data['price_diff'], '价格一阶差分序列')


fig, axes = plt.subplot(3,1,figsize=(15,12))

# 原始价格
axes[0].plot(stock_data['price'], color='blue', linewidth=1.5)
axes[0].set_title('原始价格序列',fontsize=14,fontweight='bold')
axes[0].set_ylabel('价格')
axes[0].grid(True, alpha=0.3)

#收益率
axes[1].plot(stock_data['returns'], color='green', linewidth=1)
axes[1].axhline(y=0, linestyle='--', color='red', alpha=0.7)
axes[1].set_title('收益率序列',fonsize=14,fontweight='bold')
axes[1].set_ylabel('收益率')
axes[1].grid(True, alpha=0.3)

# 价格差分
axes[2].plot(stock_data['price_diff'], color='orange', linewidth=1)
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.7)
axes[2].set_title('价格一阶差分序列', fontsize=14, fontweight='bold')
axes[2].set_ylabel('价格差分')
axes[2].set_xlabel('日期')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

第四步：ACF和PACF分析

In [ ]:
print("\n4. ACF和PACF分析")
print("-" * 30)

# 选择平稳序列进行分析
analysis_data = stock_data['returns'].dropna()

# 绘制ACF和PACF图
fig, axes = plt.subplot(2,1, figsize=(15,10))

# ACF图
plot_acf(analysis_data,ax=axes[0],lags=20,title='自相关函数(ACF)')
axes[0].set_xlabel('滞后期')
axes[0].grid(True, alpha=0.3)

# PACF图
plot_pacf(analysis_data, ax=axes[1], lags=20, title='偏自相关系数(PACF)')
axes[1].set_xlabel('滞后期')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 分析ACF和PACF的截尾特征
def analyze_acf_pacf(data, max_lags=10):
    """
    分析ACF和PACF的截尾特征,建议模型参数
    """
    from statsmodels.tsa.stattools import acf, pacf

    # 计算ACF和PACF值
    acf_value = acf(data,nlags=max_lags,fft=False)
    pacf_value = pacf(data,nlags=max_lags)

    # 95%置信区间
    n = len(data)
    confidence_interval = 1.96 / np.sqrt(n)

    # 找出显著的滞后阶数
    significant_acf = [i for i in range(1,len(acf_value)) if abs(acf_value[i])>confidence_interval]
    significant_pacf = [i for i in range(1,len(pacf_value)) if abs(pacf_value)>confidence_interval]

    print(f"显著的ACF滞后阶数: {significant_acf[:5]}")  # 只显示前5个
    print(f"显著的PACF滞后阶数: {significant_pacf[:5]}")

    # 建议参数
    if len(significant_acf) > 0:
        suggested_q = min(3, max(significant_acf[:3]))
    else:
        suggested_q = 0

    if len(significant_pacf) > 0:
        suggested_p = min(3, max(significant_pacf[:3]))
    else:
        suggested_p = 0

    print(f"建议的AR阶数 (p): {suggested_p}")
    print(f"建议的MA阶数 (q): {suggested_q}")
    
    return suggested_p, suggested_q

suggested_p, suggested_q = analyze_acf_pacf(analysis_data)

第五步：ARIMA模型识别与选择

In [ ]:
print('\n5.ARIMA模型识别与选择')
print('-'*30)

# 网格化搜索最优ARIMA参数
def grid_search_arima(data,max_p=3,max_d=2,max_q=3):
    # 限制范围：避免过拟合（高阶模型参数多，拟合噪声）
    """
    网格搜索最优ARIMA参数
    """
    import itertools

    # 生成所有可能的参数组合
    p_values = range(0,max_p+1)
    d_values = range(0, max_d + 1)
    q_values = range(0, max_q + 1)

    best_aic = float('inf')  #最优模型的AIC最小
    best_params = None
    results_list = []

    print("开始网格搜索最优ARIMA参数...")
    print("参数组合评估:")

    for p,d,q in itertools.product(p_values,d_values,q_values):
        # itertools.product 生成笛卡尔积
        try:
            model = ARIMA(data, order=(p,d,q))
            fitted_model = model.fit()

            aic = fitted_model.aic
            bic = fitted_model.bic

            results_list.append({
                'params':(p,q,d),
                'AIC':aic,
                'BIC':bic
            })

            print(f"ARIMA({p},{d},{q}) - AIC: {aic:.2f}, BIC: {bic:.2f}")

            if aic < best_aic:
                best_aic = aic
                best_params = (p,d,q)

        except Exception as e:
            print(f"ARIMA({p},{d},{q}) - 拟合失败: {str(e)[:50]}")
            continue


    # 转换为DataFrame便于分析
    results_df = pd.DataFrame(results_list)

    print(f"\n最优参数: ARIMA{best_params}")
    print(f"最佳AIC值: {best_aic:.2f}")
    
    return best_params, results_df

best_params, search_result = grid_search_arima(stock_data['returns'].dropna(), max_p=3, max_d=1, max_q=3)

# 显示前十个最佳模型
print('\n前十个最佳模型(按AIC排序)')
top_models = search_result.nsmallest(10,'AIC')
for idx, row in top_models.iterrows():
    print(f"ARIMA{row['params']} - AIC: {row['AIC']:.2f}, BIC: {row['BIC']:.2f}")

第六步：模型拟合与参数估计

In [ ]:
print("\n6. 模型拟合与参数估计")
print("-" * 30)

# 使用最优参数拟合ARIMA模型
optional_model = ARIMA(stock_data['returns'].dropna(),order=best_params)
fitted_model = optional_model.fit()

# 显示模型摘要
print("最优ARIMA模型摘要:")
print(fitted_model.summary())

# 提取模型参数
params = fitted_model.params
# 拟合后，fitted_model 包含各种属性和方法
# fitted_model.params           ← 模型估计的参数值
# fitted_model.aic              ← AIC信息准则
# fitted_model.bic              ← BIC信息准则
# fitted_model.resid            ← 残差序列
for param_name, param_value in params.item():
    print(f'{param_name}:{param_value:.6f}')


第七步：模型诊断

In [ ]:
print("\n7. 模型诊断")
print("-" * 30)

# 获取残差
residuals = fitted_model.resid

# 残差基本统计
print('残差统计信息')
print(f"残差均值：{residuals.mean():.8f}")
print(f"残差标准差：{residuals.std():.8f}")
print(f"残差偏度：{residuals.skew():.8f}")
print(f"残差峰度：{residuals.kurtosis():.8f}")

fig,axes = plt.subplot(2,3,figsize=(18,12))

# 残差时序图
axes[0,0].plot(residuals.index, residuals, color='red', linewidth=1)
axes[0,0].axhline(y=0, color='black', linestyle='--', alpha=0.7)
axes[0,0].set_title('残差时序图', fontsize=14, fontweight='bold')
axes[0,0].set_ylabel('残差')
axes[0,0].grid(True, alpha=0.3)

# 残差直方图
axes[0,1].hist(residuals,bins=30,color='lightblue',alpha=0.7,edgecolor='black')
# 叠加正态分布曲线
x = np.linspace(residuals.min(),residuals.max(),100)
# 正态分布概率函数公式
normal_curve = (1 / np.sqrt(2*np.pi)*residuals.var()) * np.exp(-0.5 * ((x-residuals.mean())/residuals.std)**2)
axes[0,1].plot(x, normal_curve, 'r-', linewidth=2,label='正态分布')
axes[0,1].set_title('残差分布直方图', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('残差')
axes[0,1].set_ylabel('密度')
axes[0,1].legend()

# 残差Q-Q图
from scipy import stats